# LAB-D2-02: Backpropagation and Gradient Check

**Purpose:** Turn backpropagation into a trace of local sensitivities that can be tested independently.

**Objectives:** `OBJ-D2-04`, reinforcement of `OBJ-D2-02` and `OBJ-D2-03`  
**Estimated duration:** 45 minutes live; under 10 seconds compute  
**Prerequisites:** `LESSON-D2-03`, `ACT-D2-02`, `LAB-D2-01`; sigmoid, BCE, and batch-first dense shapes  
**Environment:** CPU only; NumPy and matplotlib; one fixed float64 example; no network or download

Workflow: **Observe -> Predict signs -> Modify -> Check numerically -> Diagnose -> Update -> Explain -> Extend**. Restart and run in order. The first commitment cell intentionally stops execution.

In [ ]:
import copy
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=8, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.22})
print(f"Python {platform.python_version()} | NumPy {np.__version__} | matplotlib {matplotlib.__version__}")
print("Runtime target: local/Colab CPU; float64 gradient checks; no GPU required.")

## Recap: Local Sensitivity Times Upstream Gradient

A forward pass stores values. A backward pass reuses those values while carrying sensitivity from loss toward earlier parameters. At each node:

$$\text{downstream sensitivity}=\text{upstream sensitivity}\times\text{local derivative}$$

A gradient is a local rate of change. It is not causal blame, global feature importance, or evidence that the objective itself is appropriate.

## Local Dataset and Fixed `2 -> 2 -> 1` Network

One example has two input features and one binary target. Examples are rows. Both hidden units and the output use sigmoid. Every value is float64 so central finite differences can serve as a precise independent check.

In [ ]:
X = np.array([[0.6, -1.2]], dtype=np.float64)
y = np.array([[1.0]], dtype=np.float64)
parameters = {
    "W1": np.array([[0.2, -0.4], [0.7, 0.3]], dtype=np.float64),
    "b1": np.array([0.1, -0.2], dtype=np.float64),
    "W2": np.array([[0.5], [-0.6]], dtype=np.float64),
    "b2": np.array([0.05], dtype=np.float64),
}
assert X.shape == (1, 2) and y.shape == (1, 1)
assert parameters['W1'].shape == (2, 2) and parameters['b1'].shape == (2,)
assert parameters['W2'].shape == (2, 1) and parameters['b2'].shape == (1,)
assert all(value.dtype == np.float64 for value in [X, y, *parameters.values()])
print("Fixed example and parameter shapes ready.")

## Predict Signs Before Arithmetic

Without calculating exact values, predict whether a small increase to each selected parameter raises or lowers loss. Use the path from that parameter to the output, the target `1`, and the current signs of downstream weights. State a sign for `dL/dW2[0,0]`, `dL/dW2[1,0]`, `dL/db2[0]`, `dL/dW1[0,0]`, and `dL/dW1[0,1]`. Mark the least-certain path.

In [ ]:
sign_predictions = {
    "dW2_0_0": "",
    "dW2_1_0": "",
    "db2_0": "",
    "dW1_0_0": "",
    "dW1_0_1": "",
    "least_certain_path_and_reason": "",
}
assert all(value.strip() for value in sign_predictions.values()), (
    "Prediction checkpoint: record every sign and one path reason before the forward reveal."
)

## Modify: Build the Manual Forward Cache

Complete stable sigmoid, clipped BCE, and `forward`. The cache must contain `X`, `Z1`, `A1`, `Z2`, and `P`. Preserve the shapes shown by the network contract.

In [ ]:
def sigmoid(values):
    # TODO: implement a stable elementwise sigmoid.
    raise NotImplementedError("TODO: implement stable sigmoid")

def binary_cross_entropy(y_true, probabilities, epsilon=1e-12):
    # TODO: return mean clipped BCE.
    raise NotImplementedError("TODO: implement clipped BCE")

def forward(X, parameters):
    # TODO: compute Z1 -> A1 -> Z2 -> P and return the full cache.
    raise NotImplementedError("TODO: build the forward cache")

In [ ]:
cache = forward(X, parameters)
initial_loss = binary_cross_entropy(y, cache['P'])
assert cache['Z1'].shape == cache['A1'].shape == (1, 2)
assert cache['Z2'].shape == cache['P'].shape == (1, 1)
assert np.all(np.isfinite([initial_loss, *cache['Z1'].ravel(), *cache['A1'].ravel(), *cache['Z2'].ravel(), *cache['P'].ravel()]))
print(f"{'node':<6} {'shape':<10} values")
for name in ['X', 'Z1', 'A1', 'Z2', 'P']:
    print(f"{name:<6} {str(cache[name].shape):<10} {cache[name].ravel()}")
print(f"Loss: {initial_loss:.8f}")

## Inspect: Forward Values on the Computational Graph

The graph shows one relationship: values move forward through affine and sigmoid operations. After backpropagation, you will add signed sensitivities to the same stages. Node width is not parameter importance.

In [ ]:
node_names = ['X', 'Z1', 'A1', 'Z2', 'P', 'L']
node_x = np.arange(len(node_names))
node_values = [X.mean(), cache['Z1'].mean(), cache['A1'].mean(), cache['Z2'].item(), cache['P'].item(), initial_loss]
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(node_x, np.zeros_like(node_x), color='#6b7280', linewidth=2, zorder=1)
ax.scatter(node_x, np.zeros_like(node_x), c=node_values, cmap='coolwarm', s=900, edgecolor='black', zorder=2)
for x_pos, name, value in zip(node_x, node_names, node_values):
    ax.text(x_pos, 0.0, f"{name}\n{value:.3f}", ha='center', va='center', fontsize=9, zorder=3)
ax.set(xlim=(-0.6, len(node_names) - 0.4), ylim=(-0.7, 0.7), title='Forward values: X -> Z1 -> A1 -> Z2 -> P -> loss')
ax.axis('off')
plt.show()

## Modify: Manual Backpropagation

Complete `backward`. For sigmoid output with BCE, begin with `dZ2 = (P - y) / B`. Continue through `W2`, apply the hidden sigmoid local derivative, and return gradients with keys matching the parameter dictionary. Do not loop over parameters or examples.

In [ ]:
def backward(cache, parameters, y_true):
    # TODO: compute dW2, db2, dW1, and db1 with batch-first operations.
    raise NotImplementedError("TODO: implement manual backpropagation")

In [ ]:
analytic_gradients = backward(cache, parameters, y)
assert set(analytic_gradients) == set(parameters)
for name in parameters:
    assert analytic_gradients[name].shape == parameters[name].shape
    assert np.all(np.isfinite(analytic_gradients[name]))
print(f"{'parameter':<10} {'gradient shape':<16} {'gradient values'}")
for name, gradient in analytic_gradients.items():
    print(f"{name:<10} {str(gradient.shape):<16} {gradient.ravel()}")

## Check: Central Finite Differences

The supplied harness perturbs one scalar parameter at a time by `+epsilon` and `-epsilon`, then estimates slope from the two losses. It is a testing instrument, not the training algorithm. Predict what a small relative error would support and what it cannot establish about model usefulness.

In [ ]:
gradient_check_prediction = {"small_error_supports": "", "small_error_does_not_establish": ""}
assert all(value.strip() for value in gradient_check_prediction.values())

In [ ]:
def finite_difference_gradients(X, y_true, parameters, epsilon=1e-6):
    numerical = {}
    for name, values in parameters.items():
        numerical[name] = np.zeros_like(values)
        for index in np.ndindex(values.shape):
            plus = {key: value.copy() for key, value in parameters.items()}
            minus = {key: value.copy() for key, value in parameters.items()}
            plus[name][index] += epsilon
            minus[name][index] -= epsilon
            loss_plus = binary_cross_entropy(y_true, forward(X, plus)['P'])
            loss_minus = binary_cross_entropy(y_true, forward(X, minus)['P'])
            numerical[name][index] = (loss_plus - loss_minus) / (2.0 * epsilon)
    return numerical

def relative_error(analytic, numerical):
    numerator = np.abs(analytic - numerical)
    denominator = np.maximum(1e-12, np.abs(analytic) + np.abs(numerical))
    return numerator / denominator

numerical_gradients = finite_difference_gradients(X, y, parameters)
gradient_check_rows = []
for name in parameters:
    errors = relative_error(analytic_gradients[name], numerical_gradients[name])
    gradient_check_rows.append((name, float(np.max(errors))))
    print(f"{name:<4} max relative error = {np.max(errors):.3e}")
max_relative_error = max(error for _, error in gradient_check_rows)
assert max_relative_error < 1e-5

## Model Detective: Two Structured Backward Bugs

Before seeing faulty source, predict the relative-error pattern for two cases: (1) the output error sign is reversed, and (2) the hidden sigmoid factor is omitted. For each case, state whether output-layer gradients, hidden-layer gradients, or both should disagree with finite differences.

In [ ]:
bug_pattern_predictions = {
    "sign_bug_output_layer": "",
    "sign_bug_hidden_layer": "",
    "missing_factor_output_layer": "",
    "missing_factor_hidden_layer": "",
    "cheapest_discriminating_check": "",
}
assert all(value.strip() for value in bug_pattern_predictions.values())

In [ ]:
def faulty_backward(cache, parameters, y_true, defect):
    B = y_true.shape[0]
    dZ2 = (cache['P'] - y_true) / B
    if defect == 'sign':
        dZ2 = -dZ2
    dW2 = cache['A1'].T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ parameters['W2'].T
    dZ1 = dA1 if defect == 'missing_hidden_sigmoid_factor' else dA1 * cache['A1'] * (1.0 - cache['A1'])
    return {"W1": cache['X'].T @ dZ1, "b1": dZ1.sum(axis=0), "W2": dW2, "b2": db2}

bug_evidence = {}
for defect in ['sign', 'missing_hidden_sigmoid_factor']:
    gradients = faulty_backward(cache, parameters, y, defect)
    bug_evidence[defect] = {name: float(np.max(relative_error(gradients[name], numerical_gradients[name]))) for name in parameters}

print(f"{'defect':<32} {'W1':>10} {'b1':>10} {'W2':>10} {'b2':>10}")
for defect, errors in bug_evidence.items():
    print(f"{defect:<32} {errors['W1']:>10.3e} {errors['b1']:>10.3e} {errors['W2']:>10.3e} {errors['b2']:>10.3e}")
assert max(bug_evidence['sign'].values()) > 1e-2
assert max(bug_evidence['missing_hidden_sigmoid_factor']['W1'], bug_evidence['missing_hidden_sigmoid_factor']['b1']) > 1e-2
assert max(bug_evidence['missing_hidden_sigmoid_factor']['W2'], bug_evidence['missing_hidden_sigmoid_factor']['b2']) < 1e-5

## Diagnose and Repair the Local Rule

For each evidence row, cite which layers agree or disagree, identify the violated local rule, and write the targeted expression that restores the invariant. Explain why the missing hidden factor does not corrupt `W2` or `b2` in this graph.

In [ ]:
bug_diagnosis = {
    "sign_bug_evidence": "",
    "sign_bug_repair_expression": "",
    "missing_factor_evidence": "",
    "missing_factor_repair_expression": "",
    "why_output_layer_is_unchanged": "",
}
assert all(value.strip() for value in bug_diagnosis.values())

## Update Check: Move in the Negative-Gradient Direction

Predict whether one small simultaneous update should lower this fixed example's loss. Complete `apply_gradients` without mutating the original parameter dictionary, then compare before and after. This local decrease does not guarantee every future batch or metric will improve.

In [ ]:
update_prediction = {"loss_direction": "", "scope_limit": ""}
assert all(value.strip() for value in update_prediction.values())

def apply_gradients(parameters, gradients, learning_rate):
    # TODO: return new arrays after subtracting learning_rate * gradient.
    raise NotImplementedError("TODO: apply a non-mutating negative-gradient update")

In [ ]:
parameter_snapshot = {name: value.copy() for name, value in parameters.items()}
updated_parameters = apply_gradients(parameters, analytic_gradients, learning_rate=0.1)
updated_loss = binary_cross_entropy(y, forward(X, updated_parameters)['P'])
print(f"Loss before: {initial_loss:.8f}")
print(f"Loss after one negative-gradient update: {updated_loss:.8f}")
assert updated_loss < initial_loss
assert all(np.array_equal(parameters[name], parameter_snapshot[name]) for name in parameters)

## Challenge: One-Parameter Perturbation Test

Choose one scalar parameter and a signed perturbation of magnitude `1e-3`. Predict the loss direction from your analytic gradient before running. Change only that scalar in a copy, then reconcile the observed loss change with `gradient * perturbation`.

In [ ]:
challenge_parameter = ""  # TODO: W1, b1, W2, or b2.
challenge_index = None  # TODO: index tuple matching that array.
challenge_delta = None  # TODO: +1e-3 or -1e-3.
challenge_prediction = ""
assert challenge_parameter in parameters and isinstance(challenge_index, tuple)
assert challenge_delta is not None and np.isclose(abs(challenge_delta), 1e-3)
assert challenge_prediction.strip()
challenge_parameters = {name: value.copy() for name, value in parameters.items()}
challenge_parameters[challenge_parameter][challenge_index] += challenge_delta
challenge_loss = binary_cross_entropy(y, forward(X, challenge_parameters)['P'])
first_order_change = analytic_gradients[challenge_parameter][challenge_index] * challenge_delta
print(f"Observed loss change: {challenge_loss - initial_loss:+.8e}")
print(f"First-order gradient prediction: {first_order_change:+.8e}")
challenge_interpretation = ""  # TODO: compare signs and explain approximation error.
assert challenge_interpretation.strip()

## Optional Extension: Finite-Difference Epsilon Sweep

Predict why extremely large and extremely small epsilon values can both weaken a numerical gradient check. Sweep the supplied values for one selected scalar and plot relative error. This extension is independent of the core checkpoint.

In [ ]:
optional_parameter = None  # TODO (optional): parameter name.
optional_index = None  # TODO (optional): index tuple.
optional_prediction = ""
if optional_parameter is None or optional_index is None or not optional_prediction.strip():
    print("Optional epsilon sweep skipped. Core checkpoint is unaffected.")
else:
    epsilons = np.logspace(-2, -10, 9)
    errors = []
    for epsilon in epsilons:
        estimate = finite_difference_gradients(X, y, parameters, epsilon)[optional_parameter][optional_index]
        analytic_value = analytic_gradients[optional_parameter][optional_index]
        errors.append(float(relative_error(np.array(analytic_value), np.array(estimate))))
    fig, ax = plt.subplots()
    ax.loglog(epsilons, errors, marker='o', color='#1f6f8b')
    ax.set(xlabel='finite-difference epsilon', ylabel='relative error', title='Truncation and rounding trade-off')
    ax.invert_xaxis()
    plt.show()
    optional_interpretation = ""  # TODO (optional): identify the useful region and both failure modes.
    assert optional_interpretation.strip()

## Reflect and Checkpoint

Explain how sign prediction, analytic/numerical agreement, structured bug patterns, and the negative-gradient update test different claims. State one situation where finite differences could mislead and why backpropagation reuses local work more efficiently.

In [ ]:
reflection = {
    "four_checks_test_different_claims": "",
    "finite_difference_limit": "",
    "backprop_reuse": "",
}
assert all(value.strip() for value in reflection.values())
assert max_relative_error < 1e-5
assert updated_loss < initial_loss
assert max(bug_evidence['sign'].values()) > 1e-2
assert max(bug_evidence['missing_hidden_sigmoid_factor']['W1'], bug_evidence['missing_hidden_sigmoid_factor']['b1']) > 1e-2
assert max(bug_evidence['missing_hidden_sigmoid_factor']['W2'], bug_evidence['missing_hidden_sigmoid_factor']['b2']) < 1e-5
print("LAB-D2-02 checkpoint passed: float64 cache, gradient agreement, distinct bug signatures, and loss-lowering update.")

## Takeaways

- Backpropagation composes local derivatives with upstream sensitivity.
- Predicting signs exposes conceptual errors before detailed arithmetic.
- Central finite differences make the backward pass a falsifiable claim.
- Layer-local mismatch patterns help locate missing factors.
- `backward` computes gradients; a separate negative-gradient update changes parameters.

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| Output shapes become `(1,)` | A bias or probability was squeezed | Preserve `(B, 1)` through the binary output |
| Every analytic gradient has opposite sign | `P - y` was reversed | Recheck the BCE-with-sigmoid output derivative |
| Only `W1` and `b1` fail | Hidden sigmoid local derivative is missing | Multiply `dA1` by `A1 * (1 - A1)` |
| Relative error is large everywhere | Forward loss or parameter copy is inconsistent | Check float64, central differences, and non-mutating perturbations |
| Relative error worsens at tiny epsilon | Floating-point cancellation dominates | Return to the supplied calibrated epsilon |
| One update raises loss | Update sign or rate is wrong | Verify subtraction and retry the supplied small rate |

## Continue

Return to the [LAB-D2-02 debrief](../student-guide/day-2-student-guide.md#lab-d2-02---backpropagation-and-gradient-check). Review [LESSON-D2-03](../student-guide/day-2-student-guide.md#lesson-d2-03---computational-graphs-and-backpropagation), [ACT-D2-02](../challenges/day-2-challenges.md#act-d2-02---learning-rate-trajectory-and-gradient-sign), and [LAB-D2-01](LAB-D2-01-loss-learning-rate.ipynb) as needed.